# Contasis RAG + API + Ollama SmolLM2 135M

Este notebook consume la **API FastAPI del Docker**. El flujo es `Notebook/cliente → API → RAG → Ollama → código`.

La salida útil es exclusivamente el código: `01`, `6` o `6365095|4212`.

## Funcionamiento completo de la API

La API expone el motor RAG mediante FastAPI. Su propósito no es redactar explicaciones, sino convertir un texto en un **código Contasis válido**. La documentación interactiva está disponible en `http://localhost:8000/docs`.

### Flujo de una consulta

1. El cliente envía un texto a `POST /v1/codigo` o `GET /v1/codigo`.
2. La API normaliza el texto y determina el tipo de búsqueda según `modo`.
3. Para comprobantes e identidades consulta primero los catálogos exactos.
4. Para una glosa contable, el RAG recupera hasta ocho candidatos del histórico y del Plan Contasis, priorizando `empresa` y `registro`.
5. Una coincidencia histórica exacta se devuelve directamente. Si no existe, SmolLM2 selecciona entre los candidatos recuperados cuando `use_llm=true`.
6. La selección del modelo se acepta únicamente si pertenece a la lista de códigos permitidos. Si el modelo falla, está desactivado o devuelve algo inválido, se utiliza el mejor candidato recuperado como fallback.
7. La API responde como `text/plain` con el código y nada más.

`Cliente → FastAPI → catálogo o recuperación RAG → SmolLM2 (selector opcional) → validación → código`

### Endpoints

| Método y ruta | Uso | Respuesta |
|---|---|---|
| `GET /health` | Comprueba la carga del RAG, la conexión con Ollama y la disponibilidad del modelo | JSON de diagnóstico |
| `POST /v1/codigo` | Endpoint principal; recibe los parámetros en JSON | Código en `text/plain` |
| `GET /v1/codigo` | Consulta rápida; recibe los mismos parámetros en la URL | Código en `text/plain` |

### Parámetros de `/v1/codigo`

| Campo | Requerido | Valor predeterminado | Descripción |
|---|---:|---|---|
| `texto` | Sí | — | Tipo de documento, tipo de identidad o glosa que se desea clasificar |
| `modo` | No | `auto` | `auto`, `comprobante`, `identidad` o `cuenta` |
| `empresa` | No | `null` | Prioriza registros históricos de una empresa, por ejemplo `RC CORPORACION` |
| `registro` | No | `COMPRA` | Contexto contable: `COMPRA` o `VENTA` |
| `use_llm` | No | `true` | Permite que SmolLM2 elija entre candidatos; no permite inventar códigos |

### Modos y formato de salida

- `comprobante`: devuelve un código como `01` (factura) o `03` (boleta).
- `identidad`: devuelve un código como `6` (RUC) o `1` (DNI).
- `cuenta`: devuelve `CUENTA_BASE|CUENTA_TOTAL`, por ejemplo `6365095|4212`.
- `auto`: intenta reconocer primero comprobantes e identidades; si no encuentra una coincidencia, procesa el texto como glosa contable.

`CUENTA_BASE` representa la cuenta recuperada para la operación y `CUENTA_TOTAL` la cuenta total asociada al contexto de compra o venta. Los valores predeterminados globales son `4212` para compras y `1212` para ventas, salvo que el histórico de la empresa indique otro valor.

### Direcciones según el lugar de ejecución

- Notebook abierto en Jupyter dentro de Docker: `http://api:8000`.
- Notebook, PowerShell u otro cliente ejecutado en Windows: `http://localhost:8000`.

La variable de entorno `CONTASIS_API_URL` permite seleccionar automáticamente la dirección correcta. Si una petición es inválida, FastAPI responde con un error de validación; si el motor no puede resolverla por un fallo interno, responde con HTTP `500` y el mensaje `No se pudo resolver el código`.

In [1]:
import os, requests
API_URL = os.getenv('CONTASIS_API_URL', 'http://localhost:8000').rstrip('/')
print('API:', API_URL)
print(requests.get(f'{API_URL}/health', timeout=10).json())

API: http://localhost:8000
{'status': 'ok', 'rag_loaded': True, 'ollama': True, 'model': 'contasis-smollm2', 'model_available': True, 'historicos': 3212, 'plan_cuentas': 2881}


In [ ]:
def codigo(texto, modo='auto', empresa=None, registro='COMPRA', use_llm=True):
    payload = {
        'texto': texto, 'modo': modo, 'empresa': empresa,
        'registro': registro, 'use_llm': use_llm,
    }
    r = requests.post(f'{API_URL}/v1/codigo', json=payload, timeout=40)
    r.raise_for_status() #API error handling
    return r.text.strip()

## 1. Códigos de comprobante e identidad

In [3]:
print(codigo('Se trata del registro de una factura'))       # 01
print(codigo('boleta'))                                     # 03
print(codigo('RUC', modo='identidad'))                      # 6
print(codigo('DNI', modo='identidad'))                      # 1

01
03
6
1


## 2. Glosas reales: CUENTA_BASE|CUENTA_TOTAL

In [8]:
glosas = [
    'SERVICIO DE INTERNET 993630309',                # 6365095|4212
    'ENERGIA ELECTRICA JR.1RO DE NOVIEMBRE',         # 6361095|4212
    'ENERGIA ELECTRICA JR.LOS INCAS',                # 6361095|4212
    'ENERGIA ELECTRICA JR.FCO.IRAZOLA',              # 6361095|4212

    'POR EL SERVICIO DE ENERGIA ELECTRICA',          # 6361095|4212
    'POR EL SERVICIO DE AGUA POTABLE',               # 6363095|4212
    'POR LA COMPRA DE COMBUSTIBLE',                  # 656009401|4212
    'POR LA COMISION BANCARIA',                      # 6391094|4212
    'POR EL SERVICIO DE PUBLICIDAD',                 # 6371095|4212
    'POR CONSUMO DE ALIMENTOS',                      # 6314095|4212

    # Variaciones para comprobar que el RAG generaliza
    'PAGO DE SERVICIO DE INTERNET',                  # esperado: 6365095|4212
    'CONSUMO DE INTERNET DEL LOCAL',                 # esperado: 6365095|4212
    'RECIBO DE ENERGIA ELECTRICA DEL LOCAL',         # esperado: 6361095|4212
    'PAGO DE LUZ DEL ESTABLECIMIENTO',               # esperado: 6361095|4212
    'SERVICIO DE AGUA POTABLE DEL LOCAL',            # esperado: 6363095|4212
    'PAGO POR CONSUMO DE AGUA',                      # esperado: 6363095|4212
    'COMPRA DE COMBUSTIBLE PARA VEHICULO',           # esperado: 656009401|4212
    'COMISION COBRADA POR EL BANCO',                 # esperado: 6391094|4212
    'SERVICIO DE PUBLICIDAD Y DIFUSION',             # esperado: 6371095|4212
    'CONSUMO DE ALIMENTOS',                          # esperado: 6314095|4212
]

for g in glosas:
    print(
        f"{g:<55} -> "
        f"{codigo(g, empresa='RC CORPORACION', registro='COMPRA')}"
    )

SERVICIO DE INTERNET 993630309                          -> 6365095|4212
ENERGIA ELECTRICA JR.1RO DE NOVIEMBRE                   -> 6361095|4212
ENERGIA ELECTRICA JR.LOS INCAS                          -> 6361095|4212
ENERGIA ELECTRICA JR.FCO.IRAZOLA                        -> 6361095|4212
POR EL SERVICIO DE ENERGIA ELECTRICA                    -> 6361095|4212
POR EL SERVICIO DE AGUA POTABLE                         -> 6363095|4212
POR LA COMPRA DE COMBUSTIBLE                            -> 656009401|4212
POR LA COMISION BANCARIA                                -> 6391094|4212
POR EL SERVICIO DE PUBLICIDAD                           -> 6371095|4212
POR CONSUMO DE ALIMENTOS                                -> 6314095|4212
PAGO DE SERVICIO DE INTERNET                            -> 6365093|4212
CONSUMO DE INTERNET DEL LOCAL                           -> 6365093|4212
RECIBO DE ENERGIA ELECTRICA DEL LOCAL                   -> 6361093|4212
PAGO DE LUZ DEL ESTABLECIMIENTO                         -> 639

In [15]:
glosas_venta_rv = [
    'VENTA DE MERCADERIA',                       # 70121|1212
    'VENTA LOCAL DE MERCADERIAS',                # 70121|1212
    'VENTA DE MERCADERIA',                       # esperado: 70121|1212
    'VENTA DE MERCADERIA AL CONTADO',            # esperado: 70121|1212
]
    
for g in glosas_venta_rv:
    print(
        f"{g:<50} -> "
        f"{codigo(g, empresa='RV CORPORACION', registro='VENTA')}"
    )

VENTA DE MERCADERIA                                -> 70121|1212
VENTA LOCAL DE MERCADERIAS                         -> 70121|1212
VENTA DE MERCADERIA                                -> 70121|1212
VENTA DE MERCADERIA AL CONTADO                     -> 70121|1212


## 3. Consulta libre

SmolLM2 **no puede inventar cuentas**: solo escoge entre códigos reales recuperados del histórico/Plan Contasis.

In [5]:
consulta = 'SERVICIO DE INTERNET PARA OFICINA DE VENTAS'
print(codigo(consulta, modo='cuenta', empresa='RC CORPORACION', registro='COMPRA'))

6365094|4212


## 4. Llamada HTTP equivalente

La API principal es `POST /v1/codigo` y devuelve `text/plain`, no una explicación JSON.

In [7]:
payload = {'texto':'factura','modo':'auto','registro':'COMPRA','use_llm':True}
requests.post(f'{API_URL}/v1/codigo', json=payload).text

'01'